# Chapter 01: The Information Selection Problem

Offline, deterministic walkthrough of the tiny term-overlap scanner. Retrieved, eligible, selected and the evidence chain are kept separate.

In [ ]:
import sys
from pathlib import Path

# Resolve the repository root by walking parents from the current working directory.
ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

# Make this notebook's shared helper importable, then re-resolve through it.
NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)

## Load the toy corpus and inspect tokenisation

In [ ]:
import json
import keyword_search as ks

corpus_path = ROOT / "examples" / "fixtures" / "toy-v1.json"
corpus = ks.load_corpus(corpus_path)
print("snapshot:", corpus["snapshot_id"], "passages:", len(corpus["passages"]))
for passage in corpus["passages"]:
    print(passage["passage_id"], sorted(ks.terms(passage["text"])))

## Exact query: retrieved contains more than selected

In [ ]:
result = ks.search(corpus, "R17 blue button", limit=1)
print("status:", result["status"])
for candidate in result["retrieved"]:
    print(candidate["passage_id"], "score=", candidate["score"], "matched=", candidate["matched"])
print("selected:", [item["passage_id"] for item in result["selected"]])

assert result["selected"][0]["passage_id"] == "device-reset"
assert [item["passage_id"] for item in result["retrieved"]] == ["device-reset", "network-reset"]

## Paraphrase: available evidence, no candidates

In [ ]:
paraphrase = ks.search(corpus, "restore factory settings", limit=1)
print("status:", paraphrase["status"], "retrieved:", paraphrase["retrieved"])
assert paraphrase["status"] == "no_candidates"
assert paraphrase["retrieved"] == []

## Ambiguous tie: deterministic order is not intent

In [ ]:
ambiguous = ks.search(corpus, "R17 reset", limit=1)
print([(item["passage_id"], item["score"]) for item in ambiguous["retrieved"]])
assert [item["passage_id"] for item in ambiguous["retrieved"]] == ["device-reset", "network-reset"]
assert all(item["score"] == 2 for item in ambiguous["retrieved"])
assert ambiguous["selected"][0]["passage_id"] == "device-reset"

## No-answer with overlap: a candidate is not an answer

In [ ]:
overlap = ks.search(corpus, "R17 battery replacement", limit=1)
print("selected:", overlap["selected"][0]["passage_id"], "status:", overlap["status"])
assert overlap["selected"][0]["passage_id"] == "device-reset"
assert overlap["status"] == "candidates"

## Retrieved versus selected

In [ ]:
# Retrieved is the full candidate list; selected is only the top-k prefix.
exact = ks.search(corpus, "R17 blue button", limit=1)
print("available:", exact["available"])
print("retrieved:", [item["passage_id"] for item in exact["retrieved"]])
print("eligible:", exact["eligible"])
print("selected:", [item["passage_id"] for item in exact["selected"]])
print("admitted/used/helpful:", exact["admitted"], exact["used"], exact["helpful"])

## Try it yourself
Change the query or `limit` and rerun. Keep the distinction between `retrieved` and `selected` visible.

In [ ]:
for query in ["R17", "blue", "export", "Q99 calibration"]:
    r = ks.search(corpus, query, limit=2)
    print(repr(query), "->", [(c["passage_id"], c["score"]) for c in r["selected"]])

## Observed versus not established
Observed: lexical matches, misses, tie order and scope behaviour in this tiny fixture. Not established: relevance, permission beyond the trusted label, admission, use or helpfulness.